<a href="https://colab.research.google.com/github/Khaledblel/player-market-value-prediction/blob/main/00_Data_Preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 00. Data Preparation & Relational Merging
**Project:** Predicting Football Player Market Values  
**Author:** Khaled Blel (4 Data Science 2)  

In this notebook, we extract, clean, and merge multiple relational tables from the Transfermarkt dataset to build our master analytical dataframe. To ensure a robust foundation and strictly prevent temporal data leakage, we will:
1.  Establish the target variable spine using historical snapshots (`player_valuations.csv`) and carefully join static player demographics (`players.csv`) and league context (`competitions.csv`).
2.  Engineer critical point-in-time features, such as the exact `age_at_valuation` for every historical record.
3.  Calculate backward-looking performance metrics (Recent Form & Career Pedigree) using match data (`appearances.csv`) via advanced point-in-time joins (`merge_asof`), ensuring the model only learns from past events.



## 1. Environment Setup & Imports
In this section, we import the necessary libraries for data manipulation, date handling, and visualization and configure the notebook.

In [ ]:
!pip install -q kagglehub

import pandas as pd
import numpy as np
import warnings
import kagglehub
import os
from datetime import datetime

# Formatting and notebook config
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

print("Libraries imported successfully!")

Libraries imported successfully!


---
## 2. Loading the dataset
In this section, we download the dataset.

In [ ]:
data_path_cache = kagglehub.dataset_download("davidcariboo/player-scores/versions/649")
print(f"Dataset downloaded stored at: {data_path_cache}")
data_path = data_path_cache + "/"

100%|██████████| 196M/196M [00:01<00:00, 168MB/s]

Extracting files...


Dataset downloaded stored at: /root/.cache/kagglehub/datasets/davidcariboo/player-scores/versions/649


---
## 3. Loading the Spine & Player Profiles and conversion of date to datetime objects
Our objective is to build a dataset where each row is a historical snapshot of a player's market value.
*   **The Spine:** `player_valuations.csv` (contains the `date` and `market_value_in_eur`).
*   **Player Profiles:** `players.csv` (contains static demographics like date of birth, position, height).

In [ ]:
# 1. Load the spine (Target variable and dates)
valuations_df = pd.read_csv(data_path + "player_valuations.csv")
valuations_df['date'] = pd.to_datetime(valuations_df['date']) # Convert to datetime objects

# 2. Load the player demographics
players_df = pd.read_csv(data_path + "players.csv")
# Convert date of birth to datetime
players_df['date_of_birth'] = pd.to_datetime(players_df['date_of_birth'], errors='coerce')

print(f"Valuations loaded: {valuations_df.shape[0]} rows.")
print(f"Players loaded: {players_df.shape[0]} rows.")

display(valuations_df.head(3))

Valuations loaded: 526185 rows.
Players loaded: 37579 rows.


,player_id,date,market_value_in_eur,current_club_name,current_club_id,player_club_domestic_competition_id
0,405973,2000-01-20,150000,Unknown,3057.000,BE1
1,342216,2001-07-20,100000,Unknown,1241.000,SC1
2,3132,2003-12-09,400000,Dynamo Kyiv,126.000,TR1


---
## 4. Merging Demographics & Feature Engineering (Age)
We will now join the player demographics to our valuation spine.

**Critical Step (Preventing Data Leakage):** We will only select static features from `players.csv` (like position, foot, height, and date of birth). We strictly exclude columns like `highest_market_value_in_eur` or `current_club_id` because they represent future or current states that the model shouldn't know about at the time of a historical valuation.

After merging, we engineer a highly predictive feature: `age_at_valuation`.

In [ ]:
# 1. Select ONLY the static, non-leaky columns from players_df
player_cols_to_keep =[
    'player_id', 'date_of_birth', 'position', 'sub_position',
    'foot', 'height_in_cm', 'country_of_citizenship'
]
players_clean = players_df[player_cols_to_keep]

# 2. Merge with the valuations spine using a Left Join
# This ensures we keep every valuation snapshot, attaching player info where available
df = valuations_df.merge(players_clean, on='player_id', how='left')

# 3. Feature Engineering: Calculate age at the exact time of valuation
# We subtract the birth date from the valuation date, get the days, and divide by 365.25 (accounting for leap years)
df['age_at_valuation'] = (df['date'] - df['date_of_birth']).dt.days / 365.25

# 4. Clean up: Drop 'date_of_birth' as we've extracted the useful information from it
df.drop(columns=['date_of_birth'], inplace=True)

# 5. Drop rows where age couldn't be calculated (missing birth dates) or is unrealistic
df = df.dropna(subset=['age_at_valuation'])
df = df[(df['age_at_valuation'] >= 13) & (df['age_at_valuation'] <= 50)]

print(f"Dataset size after merge and age filtering: {df.shape[0]} rows.")
display(df[['player_id', 'date', 'market_value_in_eur', 'position', 'age_at_valuation']].head(3))

Dataset size after merge and age filtering: 524838 rows.


,player_id,date,market_value_in_eur,position,age_at_valuation
2,3132,2003-12-09,400000,Midfield,23.748
3,6893,2003-12-15,900000,Defender,20.099
4,10,2004-10-04,7000000,Attack,26.322


---
## 5. Adding League Context
The prestige of the league a player competes in heavily influences their market value. We will bring in the `competitions.csv` table to add this context.

We will join this data using the `player_club_domestic_competition_id` from our valuations spine, matching it to the `competition_id` in the competitions table. We will extract the league's `sub_type` (e.g., first_tier, second_tier) and whether it is a major national league.

In [ ]:
# 1. Load the competitions dataset
competitions_df = pd.read_csv(data_path + "competitions.csv")

# 2. Select only the features we need to avoid clutter
comp_cols_to_keep =[
    'competition_id',
    'sub_type', # e.g., 'first_tier', 'second_tier'
    'country_name',
    'is_major_national_league'
]
competitions_clean = competitions_df[comp_cols_to_keep]

# 3. Merge with our main dataframe
# The valuations spine uses 'player_club_domestic_competition_id' to represent the league
df = df.merge(
    competitions_clean,
    left_on='player_club_domestic_competition_id',
    right_on='competition_id',
    how='left'
)

# 4. Clean up: Drop the redundant ID columns we no longer need for modeling
df.drop(columns=['player_club_domestic_competition_id', 'competition_id'], inplace=True)

# 5. Fill missing categorical values with 'Unknown' just in case some leagues are missing from the mapping
df['sub_type'] = df['sub_type'].fillna('Unknown')
df['country_name'] = df['country_name'].fillna('Unknown')
df['is_major_national_league'] = df['is_major_national_league'].fillna(False)

print(f"Dataset shape after adding competitions: {df.shape}")
display(df[['player_id', 'market_value_in_eur', 'sub_type', 'is_major_national_league', 'country_name']].head(5))

Dataset shape after adding competitions: (524838, 14)


,player_id,market_value_in_eur,sub_type,is_major_national_league,country_name
0,3132,400000,first_tier,False,Turkey
1,6893,900000,first_tier,True,England
2,10,7000000,first_tier,True,Italy
3,26,1500000,first_tier,True,Germany
4,65,8000000,first_tier,False,Greece


---
## 6. Feature Engineering: Recent Form & Career Pedigree
Here, we process the `appearances.csv` file. To capture both a player's immediate momentum and their established reputation, we will engineer two sets of features:
1.  **Recent Form:** Rolling sum of stats (goals, assists, minutes, cards) over the last 365 days.
2.  **Career Pedigree:** Cumulative lifetime stats up to the valuation date.

In [ ]:
# 1. Load appearances and parse dates
appearances_df = pd.read_csv(data_path + "appearances.csv")
appearances_df['date'] = pd.to_datetime(appearances_df['date'])

# The stats we want to track
stat_cols =['minutes_played', 'goals', 'assists', 'yellow_cards', 'red_cards']

# 2. Aggregate stats by player and by day (in case a player has 2 entries on the same day due to data quirks)
app_daily = appearances_df.groupby(['player_id', 'date'])[stat_cols].sum().reset_index()
app_daily = app_daily.sort_values(by=['player_id', 'date'])

# 3. Calculate CAREER PEDIGREE (Cumulative stats over time)
for col in stat_cols:
    app_daily[f'career_{col}'] = app_daily.groupby('player_id')[col].cumsum()

# 4. Calculate RECENT FORM (Rolling 365 days stats)
# To use time-based rolling, we must set the date as the index and sort it
app_daily_indexed = app_daily.set_index('date').sort_index()
rolling_365d = app_daily_indexed.groupby('player_id')[stat_cols].rolling('365D').sum().reset_index()

# Rename columns for the rolling stats
rename_dict = {col: f'recent_365d_{col}' for col in stat_cols}
rolling_365d.rename(columns=rename_dict, inplace=True)

# 5. Combine Career and Recent stats into one 'Player Match History' dataframe
app_features = pd.merge(
    app_daily[['player_id', 'date'] +[f'career_{col}' for col in stat_cols]],
    rolling_365d,
    on=['player_id', 'date']
)

# 6. MERGE WITH VALUATIONS (The Anti-Leakage Step)
# Both dataframes MUST be sorted by date for merge_asof to work
df = df.sort_values('date')
app_features = app_features.sort_values('date')

df = pd.merge_asof(
    df,
    app_features,
    on='date',
    by='player_id',
    direction='backward', # Look strictly backwards in time
    allow_exact_matches=False # Exclude matches played ON the valuation day
)

# 7. Fill NaNs with 0 (If a player had no matches before a valuation, their stats are 0)
stat_feature_cols =[f'career_{col}' for col in stat_cols] + [f'recent_365d_{col}' for col in stat_cols]
df[stat_feature_cols] = df[stat_feature_cols].fillna(0)

print(f"Dataset shape after merging performance stats: {df.shape}")
display(df[['player_id', 'date', 'market_value_in_eur', 'recent_365d_goals', 'career_goals']].tail(5))

Dataset shape after merging performance stats: (524838, 24)


,player_id,date,market_value_in_eur,recent_365d_goals,career_goals
524833,636237,2026-03-23,2500000,0.000,4.000
524834,628473,2026-03-23,250000,0.000,0.000
524835,623318,2026-03-23,100000,0.000,0.000
524836,678503,2026-03-23,800000,0.000,0.000
524837,1447744,2026-03-23,100000,0.000,0.000


---
## 7. Final Data Cleaning & Missing Value Imputation
In this step, we will handle any remaining missing values (`NaN`s) in our dataset.
*   **Target Variable:** Any row missing the `market_value_in_eur` will be dropped, as it's our target.
*   **Numerical Features:** Missing `height_in_cm` will be filled with the median height.
*   **Categorical Features:** Missing `foot` or `position` will be labeled as 'Unknown'.

Once cleaned, we have our final analytical dataset ready for Exploratory Data Analysis (EDA)!

In [ ]:
# 1. Drop rows where the target variable is missing (if any)
df = df.dropna(subset=['market_value_in_eur'])

# 2. Impute missing numerical values (Height)
median_height = df['height_in_cm'].median()
df['height_in_cm'] = df['height_in_cm'].fillna(median_height)

# 3. Impute missing categorical values
df['foot'] = df['foot'].fillna('Unknown')
df['position'] = df['position'].fillna('Unknown')
df['sub_position'] = df['sub_position'].fillna('Unknown')
df['country_of_citizenship'] = df['country_of_citizenship'].fillna('Unknown')

# 4. Check if any missing values remain
missing_vals = df.isnull().sum()
print("Missing values after cleaning:")
print(missing_vals[missing_vals > 0])

df.to_csv("processed_valuations.csv", index=False)

print(f"\nFinal dataset ready! Shape: {df.shape}")
print("Dataset successfully saved as 'processed_valuations.csv'")

Missing values after cleaning:
Series([], dtype: int64)

Final dataset ready! Shape: (524838, 24)
Dataset successfully saved as 'processed_valuations.csv'
